# 02 · Classifier evaluation

Compares the three classification approaches from the implementation guide on the held-out test split: keyword baseline, embedding similarity and TF-IDF + Logistic Regression (plus a Linear SVM for reference). Metrics are also reported on the **gold subset** (seed + CUAD expert labels), which excludes heading-derived weak labels.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)

In [2]:
import json, pandas as pd
m = json.load(open('models/classifier_metrics.json'))
print({k: m[k] for k in ('trained_at','train_size','test_size','gold_test_size','embedding_backend','deployed_model')})
pd.DataFrame({k: {'accuracy (all)': v['all']['accuracy'], 'macro F1 (all)': v['all']['macro_f1'],
                  'accuracy (gold)': v['gold_only']['accuracy'], 'macro F1 (gold)': v['gold_only']['macro_f1']}
              for k, v in m['methods'].items()}).T.round(3)

{'trained_at': '2026-09-22T17:24:09.548900+00:00', 'train_size': 3308, 'test_size': 827, 'gold_test_size': 646, 'embedding_backend': 'lsa-tfidf-svd', 'deployed_model': 'tfidf_logreg'}


,accuracy (all),macro F1 (all),accuracy (gold),macro F1 (gold)
keyword_baseline,0.658,0.699,0.605,0.604
embedding_similarity,0.498,0.534,0.433,0.415
tfidf_logreg,0.877,0.878,0.882,0.812
tfidf_linear_svm,0.886,0.886,0.896,0.827


## Per-class results of the deployed model (gold subset)

In [3]:
pc = m['methods']['tfidf_logreg']['gold_only']['per_class']
pd.DataFrame({k: v for k, v in pc.items() if isinstance(v, dict) and 'support' in v}).T.round(3)

,precision,recall,f1-score,support
Payment,0.780,0.889,0.831,72.0
Confidentiality,0.667,0.400,0.500,5.0
Termination,0.756,0.800,0.777,85.0
Liability,0.972,0.775,0.862,89.0
Dispute,0.975,1.000,0.987,39.0
Others,0.911,0.921,0.916,356.0
macro avg,0.843,0.798,0.812,646.0
weighted avg,0.886,0.882,0.882,646.0


## Confusion matrix (deployed model, all test rows)

In [4]:
from backend.services import classifier
from sklearn.metrics import confusion_matrix
te = pd.read_csv('data/datasets/clause_test.csv')
pred = [p[0] for p in classifier.ml_classify_batch(te['clause'].tolist())]
labels = ['Payment','Confidentiality','Termination','Liability','Dispute','Others']
pd.DataFrame(confusion_matrix(te['label'], pred, labels=labels), index=labels, columns=labels)

,Payment,Confidentiality,Termination,Liability,Dispute,Others
Payment,97,0,5,2,0,11
Confidentiality,0,37,2,1,0,4
Termination,0,1,98,1,0,21
Liability,9,1,6,95,0,9
Dispute,1,0,0,0,70,0
Others,10,0,16,1,1,328


## Guide examples through the full classification step

In [5]:
examples = ['The employee shall receive a monthly salary of Rs. 50,000.',
 'The employee shall not disclose confidential information.',
 'Either party may terminate this agreement with 30 days notice.',
 'The employee shall be liable for damages caused by negligence.',
 'Any dispute shall be resolved through arbitration in Kochi.',
 'The employee shall work from 9:30 am to 6 pm on weekdays.']
pd.DataFrame([{'clause': t, 'category': r['category'], 'confidence': r['confidence'], 'method': r['method'],
  'keyword': r['details']['keyword']['label'], 'embedding': r['details']['embedding']['label']}
  for t, r in zip(examples, classifier.classify_batch([{'text': t, 'title': None} for t in examples]))])

Sentence Transformers unavailable: No module named 'sentence_transformers' — using LSA fallback


,clause,category,confidence,method,keyword,embedding
0,The employee shall receive a monthly salary of...,Payment,0.683,ml:tfidf-logreg,Payment,Payment
1,The employee shall not disclose confidential i...,Confidentiality,0.997,ml:tfidf-logreg,Confidentiality,Confidentiality
2,Either party may terminate this agreement with...,Termination,0.971,ml:tfidf-logreg,Termination,Termination
3,The employee shall be liable for damages cause...,Liability,0.938,ml:tfidf-logreg,Liability,Others
4,Any dispute shall be resolved through arbitrat...,Dispute,0.946,ml:tfidf-logreg,Dispute,Dispute
5,The employee shall work from 9:30 am to 6 pm o...,Others,0.539,ml:tfidf-logreg,Others,Others


## Observations
- The trained TF-IDF + Logistic Regression model clearly outperforms both baselines; it is deployed because it gives calibrated probabilities (used as confidence and for the low-confidence heading fallback).
- The embedding-similarity baseline is weak with the LSA fallback embedder; re-run this notebook after installing Sentence Transformers (`scripts/download_models.py`) and retraining to compare.
- Weakest gold-subset classes should be read from the per-class table above; small support (e.g. Confidentiality) makes those numbers less stable.